In [16]:
import pandas as pd
import numpy as np
import os
OUTPUT_DIR = "brat_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [9]:
cases = pd.read_excel("annotations_ace_confirmed_cases_v3.xlsx")
print(len(cases.groupby('ehr_id')))
print(cases.columns)

89
Index(['ehr_id', 'concept_id', 'span', 'negated', 'childhood_cue',
       'childhood_term', 'adult_cue', 'adult_term', 'frequency_cue',
       'frequency_term', 'past_t_cue', 'past_t_term', 'recent_cue',
       'recent_term', 'current_cue', 'current_term', 'ongoing_cue',
       'ongoing_term', 'perpetrator_cue', 'perpetrator_term', 'ehr_history',
       'start_char', 'end_char'],
      dtype='str')


In [11]:
cases['perpetrator_term'].unique()

<StringArray>
['padre de su hijo',                nan,            'padre',
    'marido de tia',            'amigo',            'madre',
               'ex',           'abuelo',            'prima',
           'pareja',           'marido',        'profesora',
          'hermano',           'vecino']
Length: 14, dtype: str

In [ ]:
def adjust_offset(text, offset):
    """
    Adjust offset from CRLF -> LF
    Each \r\n becomes \n, so we subtract number of CRLF occurrences before offset
    """
    shift = text[:offset].count("\r\n")
    return offset - shift

# TODO: refine with cases['prepetrator_term'].unique()
def map_perpetrator(text):
    text = str(text).lower()

    if   any(w in text for w in ["padre", "madre", "abuelo", "tio", "hermano"]):  return "family_member"
    elif any(w in text for w in ["novio", "pareja", "marido", "ex"]):             return "partner"
    elif "amig" in text:                                                          return "friend"
    elif any(w in text for w in ["vecino", "profesor"]):                          return "known_other"
    elif text.strip():                                                            return "unknown_other"
    else:                                                                         return "unspecified"

def normalize_row(row):

    attrs = {}

    # Negation → only if true
    if   row["negated"]:                    attrs["Negation"] = "true"

    # Childhood → only if present
    if   pd.notna(row["childhood_term"]):   attrs["Childhood"] = "true"

    # Adult → only if present
    if   pd.notna(row["adult_term"]):       attrs["Adult"] = "true"

    # Temporality
    if   pd.notna(row["past_t_term"]):      attrs["Temporality"] = "past"
    elif pd.notna(row["recent_term"]):      attrs["Temporality"] = "recent"
    elif pd.notna(row["current_term"]):     attrs["Temporality"] = "current"
    elif pd.notna(row["ongoing_term"]):     attrs["Temporality"] = "ongoing"

    # Frequency
    if pd.notna(row["frequency_term"]): 
        # your mapping logic here
        attrs["Frequency"] = "repeated"  # example

    # Perpetrator
    if pd.notna(row["perpetrator_term"]):  attrs["Perpetrator"] = map_perpetrator(row["perpetrator_term"])

    return attrs

def to_brat_ann(df):

    for ehr_id, group in df.groupby("ehr_id"):

        original_text = group.iloc[0]["ehr_history"]

        # --- Normalize line endings to LF
        text = original_text.replace("\r\n", "\n")

        txt_path = os.path.join(OUTPUT_DIR, f"caso_{ehr_id}.txt")
        ann_path = os.path.join(OUTPUT_DIR, f"caso_{ehr_id}.ann")

        # --- Write TXT (force LF)
        with open(txt_path, "w", encoding="utf-8", newline="\n") as f: 
            f.write(text)

        ann_lines = []
        t_counter = 1
        a_counter = 1

        for _, row in group.iterrows():

            # --- Adjust offsets
            start_orig = int(row["start_char"])
            end_orig = int(row["end_char"])
            start = adjust_offset(original_text, start_orig)
            end = adjust_offset(original_text, end_orig)

            # --- Extract span from normalized text
            span_text = text[start:end]

            # --- Optional validation (HIGHLY recommended)
            # Uncomment if you want strict validation:
            # if span_text != row["span"]:
            #     print(f"WARNING mismatch in {ehr_id}: {span_text} != {row['span']}")

            # --- ENTITY
            t_id = f"T{t_counter}"
            label = row["concept_id"]

            ann_lines.append(f"{t_id}\t{label} {start} {end}\t{span_text}")

            # --- ATTRIBUTES
            attrs = normalize_row(row)

            for attr_name, value in attrs.items():
                a_id = f"A{a_counter}"
                ann_lines.append(f"{a_id}\t{attr_name} {t_id} {value}")
                a_counter += 1

            t_counter += 1

        # --- Write ANN (force LF)
        with open(ann_path, "w", encoding="utf-8", newline="\n") as f:
            f.write("\n".join(ann_lines))
    

In [22]:
# run to generate .txt and .ann files for each EHR case
to_brat_ann(cases)

In [23]:
cases[cases['ehr_id'] == 216535]

,ehr_id,concept_id,span,negated,childhood_cue,childhood_term,adult_cue,adult_term,frequency_cue,frequency_term,...,recent_term,current_cue,current_term,ongoing_cue,ongoing_term,perpetrator_cue,perpetrator_term,ehr_history,start_char,end_char
0,216535,ACE_GENERAL_TRAUMA,victima,False,False,NaN,False,NaN,False,NaN,...,NaN,False,NaN,False,NaN,True,padre de su hijo,"Primer contacto con psiquiatría en el 2008, cu...",148,155
1,216535,ACE_PHYSICAL_ABUSE,malos tratos,False,False,NaN,False,NaN,False,NaN,...,NaN,False,NaN,False,NaN,True,padre de su hijo,"Primer contacto con psiquiatría en el 2008, cu...",159,171
2,216535,ACE_SEXUAL_ABUSE,abusos sexuales,False,True,en la infancia,False,NaN,False,NaN,...,NaN,False,NaN,False,NaN,False,NaN,"Primer contacto con psiquiatría en el 2008, cu...",206,221
3,216535,ACE_PARENT_SUBSTANCE,Madre alcoholica,False,True,desde pequeña,False,NaN,False,NaN,...,NaN,False,NaN,False,NaN,False,NaN,"Primer contacto con psiquiatría en el 2008, cu...",867,883
4,216535,ACE_UNSPECIFIED_NEGLECT,negligencia,False,False,NaN,False,NaN,False,NaN,...,NaN,False,NaN,False,NaN,False,NaN,"Primer contacto con psiquiatría en el 2008, cu...",1102,1113
